In [1]:
#import both chunks and embeddings

In [2]:
from pathlib import Path
import numpy as  np

In [3]:
#loading chunks again
file_path=Path("insurance_claims_records.txt")
text=file_path.read_text(encoding="utf-8")
chunks= [chunk.strip() for chunk in text.split("---") if  chunk.strip()]

print("Chunks loaded:", len(chunks))

Chunks loaded: 13001


In [4]:
#Load saved embeddings
embeddings=np.load("insurance_embedding.npy")
print("Embeddings loaded:", embeddings.shape)

Embeddings loaded: (13001, 384)


In [5]:
import faiss
print("FAISS version:",faiss.__version__)

FAISS version: 1.13.0


In [6]:
#we have to ensure correct datatypes
#this is very critical as the FAISS may misbehave wth different datatypes
embeddings=embeddings.astype("float32")
print("Embedding dtype:",embeddings.dtype)

Embedding dtype: float32


In [7]:
#Create FAISS index
dimension=embeddings.shape[1]

#FAISS needs to know:What is the size of each vector it stores
#vector search requires fixed-length vectors.
index=faiss.IndexFlatL2(dimension)
print("Index created with dimension:", dimension)
#Copies all 13001 vectors into FAISS internal memory.Makes them searchable.
#index.ntotal:Tells how many vectors are currently stored in FAISS.
index.add(embeddings)
print("Total vectors in index:",index.ntotal)

Index created with dimension: 384
Total vectors in index: 13001


In [8]:
#Load the Embedding Model (for query)
from sentence_transformers import SentenceTransformer

#The embedding model used for:Documents,Queries MUST be the same.
model= SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready for query embedding")

Model ready for query embedding


In [9]:
#next we are defining a query
query="Customer aged 45 with high insurance claim amount"

#now we need to convert tot query into embedding
query_embedding=model.encode([query]).astype("float32")
print("Query embedding shape",query_embedding.shape)


Query embedding shape (1, 384)


In [10]:
#Search Top-k Similar Records
k=5
#Find the top 5 vectors in the index that are closest to this query 
#Take the query vector.Compute L2 distance to every stored embedding.
#Sort those distances (smallest first).
distances,indices=index.search(query_embedding,k)

print("Distances:\n",distances)
print("Indices:\n",indices)

Distances:
 [[0.5365407  0.5370359  0.54199016 0.5484047  0.5494197 ]]
Indices:
 [[ 326 3985  145 5006 4501]]


We asked:"Customer aged 45 with high insurance claim amount"
FAISS:Computed distance between query vector and all 13001 vectors.
Found the 5 closest ones.
Returned their positions.

This means:
These 5 vectors are very close to the query.
They are tightly clustered in semantic space.
Retrieval is stable (no large jumps).

In [12]:
# retreive the actual recors corresponding to this indices

for idx in indices[0]:
    print("\n--- Retrieved Record ---\n")
    print(chunks[idx])


--- Retrieved Record ---

insurance record 326

This insurance record describes a customer aged 47.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Single.
The customer has an annual income of 161901.0.
The recorded insurance claim amount for this profile is 1328.0.

--- Retrieved Record ---

insurance record 3985

This insurance record describes a customer aged 45.0.
The customer is Male and works as a CEO.
Their education level is PhD and marital status is Single.
The customer has an annual income of 222003.0.
The recorded insurance claim amount for this profile is 2264.0.

--- Retrieved Record ---

insurance record 145

This insurance record describes a customer aged 45.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Married.
The customer has an annual income of 120715.0.
The recorded insurance claim amount for this profile is 80927.0.

--- Retrieved Record ---

insurance record 5006


In [13]:
#Right now your system retrieves information, but it does not answer the question.
#Next we add the Generation step.
#Then an LLM will generate the response.

In [33]:
#Next we must convert those records into one structured context block that an LLM can read.
#Store retreived records instead of printing
retrieved_chunks= [chunks[idx] for idx in indices[0]]
print("Number of retreived records :",len(retreived_chunks))

Number of retreived records : 5


In [35]:
for record in retrieved_chunks:
    print("\n--- Stored Record ---\n")
    print(record)


--- Stored Record ---

insurance record 326

This insurance record describes a customer aged 47.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Single.
The customer has an annual income of 161901.0.
The recorded insurance claim amount for this profile is 1328.0.

--- Stored Record ---

insurance record 3985

This insurance record describes a customer aged 45.0.
The customer is Male and works as a CEO.
Their education level is PhD and marital status is Single.
The customer has an annual income of 222003.0.
The recorded insurance claim amount for this profile is 2264.0.

--- Stored Record ---

insurance record 145

This insurance record describes a customer aged 45.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Married.
The customer has an annual income of 120715.0.
The recorded insurance claim amount for this profile is 80927.0.

--- Stored Record ---

insurance record 5006

This insura

In [37]:
#Build the context block, LLM expects one continuous piece of text as context.
context="\n\n".join(retrieved_chunks)
print(context[:800])

insurance record 326

This insurance record describes a customer aged 47.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Single.
The customer has an annual income of 161901.0.
The recorded insurance claim amount for this profile is 1328.0.

insurance record 3985

This insurance record describes a customer aged 45.0.
The customer is Male and works as a CEO.
Their education level is PhD and marital status is Single.
The customer has an annual income of 222003.0.
The recorded insurance claim amount for this profile is 2264.0.

insurance record 145

This insurance record describes a customer aged 45.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Married.
The customer has an annual income of 120


In [39]:
print("Context length:", len(context))

Context length: 1471


In [41]:
#Now we combine:the context (retrieved insurance records) and the user question
question=query
prompt= f"""
You are an insurace analysis assistant.
Use the following insurance records to answer the question.

Context:{context}

Question:{question}

Answer:
"""
print(prompt[:1000])


You are an insurace analysis assistant.
Use the following insurance records to answer the question.

Context:insurance record 326

This insurance record describes a customer aged 47.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Single.
The customer has an annual income of 161901.0.
The recorded insurance claim amount for this profile is 1328.0.

insurance record 3985

This insurance record describes a customer aged 45.0.
The customer is Male and works as a CEO.
Their education level is PhD and marital status is Single.
The customer has an annual income of 222003.0.
The recorded insurance claim amount for this profile is 2264.0.

insurance record 145

This insurance record describes a customer aged 45.0.
The customer is Male and works as a Doctor.
Their education level is PhD and marital status is Married.
The customer has an annual income of 120715.0.
The recorded insurance claim amount for this profile is 80927.0.

insurance record 